# Module 11 — Choosing the right statistical test (comprehensive review)
Dedicated notebook for Module 11 — re-runs 6 real research scenarios already defined on the web page, each one using the EXACT test selected by the decision tree. No NEW statistical analysis here — this is a review notebook that consolidates the syntax already learned in Modules 05-10.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

In [1]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset -- SAME seed=2026 used
    throughout Modules 03-08, matches lop_hoc_240.csv/.sav in the repo
    exactly (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

Created df: (240, 14)


In [2]:
df["gain"] = df["posttest"] - df["pretest"]

## Scenario 1 — Comparing 3 INDEPENDENT schools, parametric variable → one-way ANOVA (Module 06)

In [3]:
groups = [g["math"].values for _, g in df.groupby("school")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"F={f_stat:.3f}, p={p_val:.4f}")

F=5.875, p=0.0032


## Scenario 2 — Same subjects, measured twice, parametric → paired t-test (Module 05)

In [4]:
t, p = stats.ttest_rel(df["posttest"], df["pretest"])
print(f"t={t:.2f}, p={p:.2e}")

t=13.37, p=8.25e-31


## Scenario 3 — 3 independent groups, ordinal (Likert) variable → Kruskal-Wallis (Module 07)

In [5]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

H=2.844, p=0.241


## Scenario 4 — Measure the association (Pearson r) THEN predict (regression) (Module 08 + 09)

In [6]:
mask = df["study_hours"] > 20
df.loc[mask, "study_hours"] = df.loc[mask, "study_hours"] / 10
r, p_r = stats.pearsonr(df["study_hours"], df["math"])
print(f"Step 1 -- Pearson r: r={r:.3f}, p={p_r:.2e}")

import statsmodels.api as sm
X = sm.add_constant(df[["study_hours"]])
m = sm.OLS(df["math"], X).fit()
print(f"Step 2 -- Regression: math = {m.params['const']:.2f} + {m.params['study_hours']:.2f}×study_hours, R²={m.rsquared:.3f}")

Step 1 -- Pearson r: r=0.582, p=4.07e-23


Step 2 -- Regression: math = 35.58 + 2.51×study_hours, R²=0.338


## Scenario 5 — Finding a latent structure across MULTIPLE variables → factor analysis (Module 10)

In [7]:
items = df[["h1","h2","h3","a1","a2","a3"]].dropna()
from factor_analyzer import FactorAnalyzer
fa = FactorAnalyzer(n_factors=2, rotation="varimax")
fa.fit(items)
print(pd.DataFrame(fa.loadings_, index=items.columns, columns=["Factor1","Factor2"]).round(3))

    Factor1  Factor2
h1   -0.043    0.747
h2   -0.003    0.603
h3   -0.096    0.670
a1    0.725   -0.092
a2    0.722   -0.083
a3    0.695   -0.016


## Scenario 6 — Grouping OBJECTS (schools) with similar profiles → cluster analysis (Module 10)

In [8]:
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
vars_ = ["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP"]
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
d = truong[vars_].dropna()
X = StandardScaler().fit_transform(d)
km = KMeans(n_clusters=3, random_state=2026, n_init=10).fit(X)
print("Cluster sizes:", list(np.bincount(km.labels_)))

Cluster sizes: [np.int64(64), np.int64(49), np.int64(82)]


## Summary: quick lookup table for the 6 scenarios
| # | Research question | Method chosen | Module |
|---|---|---|---|
| 1 | 3 independent schools, Math score (parametric) | One-way ANOVA | 06 |
| 2 | Same students, pretest→posttest | Paired t-test | 05 |
| 3 | 3 independent schools, Likert rating | Kruskal-Wallis | 07 |
| 4 | Association AND prediction | Pearson r + Regression | 08, 09 |
| 5 | Latent structure across 6 items | Factor analysis (EFA) | 10 |
| 6 | Grouping 195 schools by profile | Cluster analysis (K-means) | 10 |

This is exactly the 4-question decision tree shown on the web page — applying it in order (goal → scale of measurement → number of groups/direction of association) always leads to the correct row in this table.